In [1]:
from datasets import load_dataset
#加载数据
data = load_dataset("csv", data_files="ChnSentiCorp_htl_all.csv")
#划分train/val
data = data["train"].train_test_split(test_size=0.2, seed=42)

#过滤none数据
data = data.filter(lambda x: x["review"] is not None)
data

Generating train split: 7766 examples [00:00, 83899.04 examples/s]
Filter: 100%|██████████| 1554/1554 [00:00<00:00, 56200.84 examples/s]


DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 6211
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 1554
    })
})

In [2]:
from transformers import AutoTokenizer
#加载分词器
tokenizer = AutoTokenizer.from_pretrained("rbt3")

#定义数据处理函数:文本->input_ids/attention_mask,标签->labels
def process_function(examples):
    tokenized_examples = tokenizer(examples["review"], max_length=128, truncation=True)
    tokenized_examples["labels"] = examples["label"]
    return tokenized_examples

#批量处理整个数据集
tokenized_datasets = data.map(process_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 1554/1554 [00:00<00:00, 9926.42 examples/s] 


DatasetDict({
    train: Dataset({
        features: ['label', 'review', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 6211
    })
    test: Dataset({
        features: ['label', 'review', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1554
    })
})

In [ ]:
from transformers import DataCollatorWithPadding
#数据整理器:动态padding,把每个batch补齐到该batch中最长样本(上限128),并对齐到8的倍数
collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, max_length=128, pad_to_multiple_of=8)



In [4]:
from transformers import AutoModelForSequenceClassification
#加载模型:在 rbt3 上接一个 2 分类头(0=负向, 1=正向)
model = AutoModelForSequenceClassification.from_pretrained("rbt3", num_labels=2)
model

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 12045.67it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: rbt3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly 

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, e

In [5]:
import evaluate
#加载评估指标:准确率和F1
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

#评估函数:每个epoch结束由Trainer调用,输入为(预测logits, 真实标签)
def eval_metric(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)  #取分数更高的类别
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {**accuracy, **f1}

In [6]:
from transformers import TrainingArguments
#训练参数配置
training_args = TrainingArguments(
    output_dir="./results",           # 训练输出目录(checkpoint/日志)
    num_train_epochs=3,               # 训练轮数
    per_device_train_batch_size=64,   # 训练批大小
    per_device_eval_batch_size=64,    # 评估批大小
    learning_rate=2e-5,               # 微调学习率(预训练模型惯用小值)
    warmup_steps=50,                  # 前50步学习率线性预热
    weight_decay=0.01,                # 权重衰减
    logging_steps=20,                 # 每20步打印一次日志
    eval_strategy="epoch",            # 每轮结束评估(旧版参数名 evaluation_strategy)
    save_strategy="epoch",            # 每轮结束保存checkpoint
    load_best_model_at_end=True,      # 训练结束自动加载最优checkpoint
    metric_for_best_model="f1",       # "最优"的判定指标
)

In [7]:
from transformers import Trainer
#组装Trainer:模型、参数、数据集、数据整理器、评估函数
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=collator,
    compute_metrics=eval_metric,
)

In [8]:
#开始训练(观察 loss 和每轮评估指标)
trainer.train()

d:\project\step3\llm\Lib\site-packages\transformers\tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.351369,0.309787,0.870013,0.904896
2,0.271175,0.303682,0.868726,0.901067
3,0.206647,0.285839,0.887387,0.917414


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.91it/s]
d:\project\step3\llm\Lib\site-packages\transformers\tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.07it/s]
d:\project\step3\llm\Lib\site-packages\transformers\tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.73it/s]


TrainOutput(global_step=294, training_loss=0.328456744855764, metrics={'train_runtime': 126.1453, 'train_samples_per_second': 147.711, 'train_steps_per_second': 2.331, 'total_flos': 312777705420000.0, 'train_loss': 0.328456744855764, 'epoch': 3.0})

In [9]:
#训练结束,加载的最优模型在测试集上做最终评估
trainer.evaluate()

d:\project\step3\llm\Lib\site-packages\transformers\tokenization_utils_base.py:2355: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.206647,0.285839,3,0.887387,0.917414


{'eval_loss': 0.2858385443687439,
 'eval_accuracy': 0.8873873873873874,
 'eval_f1': 0.9174138744690892}

In [10]:
import torch

#给模型换上可读的标签名
model.config.id2label = {0: "负向", 1: "正向"}
model.config.label2id = {"负向": 0, "正向": 1}

#推理函数:评论列表 -> 预测标签 + 正向概率
def predict(texts):
    model.eval()
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    for text, p in zip(texts, probs):
        label = model.config.id2label[p.argmax().item()]
        print(f"[{label}] 正向概率{p[1]:.3f} | {text}")

#几条新鲜的酒店评论,看看微调后的效果
predict([
    "房间很大很干净,前台服务特别热情,下次还会入住!",
    "早餐太差了,房间还有异味,完全不值这个价!",
    "位置还行,设施有点旧,不过价格便宜,凑合住吧。",
    "不推荐,空调噪音大得睡不着,不会再来了。",
])

[正向] 正向概率0.994 | 房间很大很干净,前台服务特别热情,下次还会入住!
[负向] 正向概率0.029 | 早餐太差了,房间还有异味,完全不值这个价!
[正向] 正向概率0.936 | 位置还行,设施有点旧,不过价格便宜,凑合住吧。
[负向] 正向概率0.114 | 不推荐,空调噪音大得睡不着,不会再来了。
